# 01 · Create the survey dataset

ChatGPT digitised Figure 6 from Japan’s Ministry of the Environment. The figure gives one result per survey square. Blue means the rail was detected. Yellow means the square was surveyed without a detection.

Run this notebook from the Day 20 folder to regenerate the CSV. Each row describes one published survey square. Its coordinates are an estimated centre based on the grid alignment, which was checked visually against the Okinawa coastline.

[Survey report](https://kyushu.env.go.jp/okinawa/press_00131.html) · [Source figure](https://kyushu.env.go.jp/okinawa/content/000377212.png)

In [1]:
import csv
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
from PIL import Image

FIGURE_URL = "https://kyushu.env.go.jp/okinawa/content/000377212.png"
DATA_FILE = Path("data/yanbaru_kuina_survey_cells.csv")


def download(url):
    request = Request(url, headers={"User-Agent": "Day20-MapChallenge/1.0"})
    with urlopen(request, timeout=90) as response:
        return response.read()

## Download and read Figure 6

Pillow opens the picture and NumPy stores each pixel as red, green and blue values. The code checks the image dimensions before using the panel calibration.

In [2]:
with Image.open(BytesIO(download(FIGURE_URL))) as figure:
    if figure.size != (900, 310):
        raise ValueError("Figure 6 dimensions changed. Review the survey grid alignment.")
    rgb = np.asarray(figure.convert("RGB"), dtype=np.int16)

## Read the survey squares

The four numbers for each panel locate the grid in the image. They give its left edge, top edge, cell width and cell height in pixels. Adding 0.5 to a row or column gives the centre of a square.

The code tests a 5-by-7 pixel patch around each centre. It checks how much of the patch looks blue or yellow. It skips the printed legend and squares without enough of either colour.

To place a square on Okinawa, start at the estimated northwest corner and move east by 0.0125° per column and south by 1/120° per row. CSV row and column numbers start at 1. The detection value is 1 for blue and 0 for yellow.

In [3]:
# Panel calibrations are left edge, top edge, cell width and cell height, in pixels.
panels = {
    2007: (18.4, 17.25, 11.895, 7.912),
    2024: (617.85, 18.55, 11.765, 7.907),
}
# Estimated northwest corner from a visual comparison with the official Okinawa coastline.
west, north = 128.0625, 26.875
lon_step, lat_step = 1 / 80, 1 / 120
csv_rows = []
summary = {}

for year, (left, top, width, height) in panels.items():
    detected_count = 0
    surveyed_count = 0
    for row in range(34):
        for col in range(23):
            if row < 7 and col < 10:
                continue  # Printed legend, outside the survey footprint.
            x = round(left + width * (col + 0.5))
            y = round(top + height * (row + 0.5))
            red, green, blue = rgb[y - 2:y + 3, x - 3:x + 4].transpose(2, 0, 1)
            blue_share = ((blue > 120) & (blue - red > 70) & (blue - green > 42)).mean()
            yellow_share = (
                (red > 140) & (green > 140) & (np.minimum(red, green) - blue > 55)
            ).mean()
            if max(blue_share, yellow_share) < 0.45:
                continue
            lon = west + (col + 0.5) * lon_step
            lat = north - (row + 0.5) * lat_step
            detected = int(blue_share > yellow_share)
            csv_rows.append((year, row + 1, col + 1, f"{lon:.7f}", f"{lat:.7f}", detected))
            detected_count += detected
            surveyed_count += 1
    summary[year] = (detected_count, surveyed_count - detected_count)

## Check and save the CSV

These totals come from the original digitisation. They help catch changes to the published image or the colour rules. The grid location remains a visual estimate.

In [4]:
expected = {2007: (102, 88), 2024: (199, 62)}
if summary != expected:
    raise ValueError(f"Check the panel alignment and colour rules. Read {summary}, expected {expected}.")

DATA_FILE.parent.mkdir(exist_ok=True)
with DATA_FILE.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "survey_year", "source_row", "source_column",
        "cell_center_lon", "cell_center_lat", "rail_detected",
    ])
    writer.writerows(csv_rows)

for year, (detected, not_detected) in summary.items():
    print(f"{year}: {detected} squares with a detection, {not_detected} without one")
print(f"Saved {len(csv_rows)} survey squares to {DATA_FILE}")

2007: 102 squares with a detection, 88 without one
2024: 199 squares with a detection, 62 without one
Saved 451 survey squares to data/yanbaru_kuina_survey_cells.csv
